# Learning to Rank — Hands-on Tutorial
## NI-ADM | Czech Technical University in Prague

In this tutorial we will:
1. **Understand** the Learning-to-Rank (LTR) problem formulation
2. **Explore** a real LTR dataset (MQ2008 from LETOR)
3. **Implement** three approaches: Pointwise, Pairwise, Listwise
4. **Evaluate** with ranking metrics: NDCG@K, MRR
5. **Experiment** with LambdaMART (LightGBM) — the industry workhorse
6. **Discuss** debiasing, feature engineering, and connection to personalized search

---

### What is Learning to Rank?

LTR is a family of ML techniques that produce an **optimal ordering** of items for a given query or user context. Unlike classification (predict a label) or regression (predict a value), the goal is to **sort** items so that the most relevant ones appear at the top.

**Applications:** search engines, recommender systems, personalized query suggestion, ad ranking, news feeds.

**Three main approaches:**

| Approach | Idea | Loss considers | Example |
|----------|------|----------------|---------|
| **Pointwise** | Predict relevance score per item independently | Single items | Logistic regression, regression to relevance |
| **Pairwise** | Learn that item A should rank above item B | Pairs of items | RankNet, RankSVM, BPR |
| **Listwise** | Directly optimize a ranking metric over the whole list | Entire list | LambdaMART, ListNet, ApproxNDCG |

## 0. Setup

In [1]:
# Install required packages (run once)
!pip install pandas scikit-learn lightgbm matplotlib seaborn -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
import os
import urllib.request
import zipfile

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('All imports OK!')

ModuleNotFoundError: No module named 'pandas'

## 1. Load the MQ2008 Dataset

**MQ2008** is a standard LTR benchmark from the [LETOR project](https://www.microsoft.com/en-us/research/project/letor-learning-rank-information-retrieval/) (Microsoft Research). It contains:
- ~800 queries
- ~15,000 query-document pairs
- 46 features per pair (TF-IDF variants, BM25, PageRank, etc.)
- Relevance labels: 0 (not relevant), 1 (partially relevant), 2 (highly relevant)

The data is in **LETOR/SVMRank format:**
```
label qid:query_id feature1:value1 feature2:value2 ...
```

This is exactly the kind of data you encounter in real search/recommendation ranking — an impression (query) with a list of candidate items (documents), each with features and a relevance signal.

In [ ]:
# Download the MQ2008 dataset
# Source: LETOR 4.0 (Microsoft Research)
# https://www.microsoft.com/en-us/research/project/letor-learning-rank-information-retrieval/
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

MQ2008_DIR = os.path.join(DATA_DIR, 'MQ2008')

def download_mq2008():
    """Try downloading MQ2008 from known mirrors."""
    urls = [
        # OneDrive shared link (LETOR 4.0 official)
        'https://api.onedrive.com/v1.0/shares/s!AtsMfWUz5l8nbOIoJ6Ks0bEMp78/root/content',
    ]
    zip_path = os.path.join(DATA_DIR, 'MQ2008.rar')
    for url in urls:
        try:
            print(f'Trying {url[:60]}...')
            urllib.request.urlretrieve(url, zip_path)
            try:
                with zipfile.ZipFile(zip_path, 'r') as z:
                    z.extractall(DATA_DIR)
                return True
            except zipfile.BadZipFile:
                print('Not a zip, trying unrar...')
                os.system(f'cd {DATA_DIR} && unrar x -o+ MQ2008.rar')
                if os.path.exists(MQ2008_DIR):
                    return True
        except Exception as e:
            print(f'  Failed: {e}')
    return False

def generate_synthetic_ltr_data():
    """Generate synthetic LTR dataset mimicking MQ2008 structure.
    Fallback if download fails (e.g., no internet in classroom)."""
    print('Generating synthetic LTR dataset (same structure as MQ2008)...')
    np.random.seed(2024)
    n_features = 46

    def make_split(n_queries, docs_range=(5, 40)):
        rows = []
        for qid in range(1, n_queries + 1):
            n_docs = np.random.randint(docs_range[0], docs_range[1])
            feats = np.random.randn(n_docs, n_features) * 0.5
            # True relevance driven by a few key features
            w = np.zeros(n_features)
            w[0] = 1.5; w[5] = 1.2; w[10] = -0.8; w[15] = 0.9; w[20] = 0.7
            w[25] = -0.5; w[30] = 0.6; w[35] = 0.4
            scores = feats @ w + np.random.randn(n_docs) * 0.8
            labels = np.zeros(n_docs, dtype=int)
            labels[scores > np.percentile(scores, 70)] = 1
            labels[scores > np.percentile(scores, 90)] = 2
            for i in range(n_docs):
                row = {f'f{j+1}': round(feats[i, j], 6) for j in range(n_features)}
                row['label'] = labels[i]
                row['qid'] = qid
                rows.append(row)
        return pd.DataFrame(rows)

    return make_split(400), make_split(100), make_split(100)

if os.path.exists(os.path.join(MQ2008_DIR, 'Fold1')):
    print('MQ2008 dataset already present.')
    USE_REAL_DATA = True
elif download_mq2008():
    print('MQ2008 downloaded successfully!')
    USE_REAL_DATA = True
else:
    print('\nCould not download MQ2008. Using synthetic data instead.')
    print('(To use real data, download MQ2008 manually from:')
    print(' https://www.microsoft.com/en-us/research/project/letor-learning-rank-information-retrieval/letor-4-0/ )')
    USE_REAL_DATA = False

In [ ]:
def parse_letor(file_path):
    """Parse LETOR/SVMRank format into a DataFrame."""
    rows = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            label = int(parts[0])
            qid = int(parts[1].split(':')[1])
            features = {}
            for part in parts[2:]:
                if ':' in part and not part.startswith('#'):
                    fid, val = part.split(':')
                    features[f'f{fid}'] = float(val)
                else:
                    break  # comment section starts
            features['label'] = label
            features['qid'] = qid
            rows.append(features)
    return pd.DataFrame(rows)


if USE_REAL_DATA:
    # Use Fold1 for our tutorial
    base = os.path.join(MQ2008_DIR, 'Fold1')
    df_train = parse_letor(os.path.join(base, 'train.txt'))
    df_val   = parse_letor(os.path.join(base, 'vali.txt'))
    df_test  = parse_letor(os.path.join(base, 'test.txt'))
    print('Loaded MQ2008 (real data)')
else:
    df_train, df_val, df_test = generate_synthetic_ltr_data()
    print('Loaded synthetic LTR data')

print(f'Train: {len(df_train):,} rows, {df_train.qid.nunique()} queries')
print(f'Val:   {len(df_val):,} rows, {df_val.qid.nunique()} queries')
print(f'Test:  {len(df_test):,} rows, {df_test.qid.nunique()} queries')
df_train.head()

## 2. Explore the Data

Before building models, let's understand what we're working with.

In [ ]:
# Relevance label distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [('Train', df_train), ('Val', df_val), ('Test', df_test)]):
    df['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#27ae60'])
    ax.set_title(f'{name} — Label distribution')
    ax.set_xlabel('Relevance label')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f"\nLabel proportions (train):")
print(df_train['label'].value_counts(normalize=True).sort_index())

In [ ]:
# How many documents per query?
docs_per_query = df_train.groupby('qid').size()
print(f'Documents per query: min={docs_per_query.min()}, median={docs_per_query.median():.0f}, '
      f'max={docs_per_query.max()}, mean={docs_per_query.mean():.1f}')

fig, ax = plt.subplots(figsize=(8, 4))
docs_per_query.hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of documents per query')
ax.set_ylabel('Number of queries')
ax.set_title('Distribution of list sizes (train)')
plt.show()

In [ ]:
# Relevant docs per query — are some queries harder?
rel_per_query = df_train.groupby('qid')['label'].apply(lambda x: (x > 0).sum())
print(f'Relevant docs per query: min={rel_per_query.min()}, median={rel_per_query.median():.0f}, '
      f'max={rel_per_query.max()}')
print(f'Queries with zero relevant docs: {(rel_per_query == 0).sum()}')

In [ ]:
# Quick look at feature correlations with relevance
feature_cols = [c for c in df_train.columns if c.startswith('f')]
correlations = df_train[feature_cols + ['label']].corr()['label'].drop('label').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
correlations.plot(kind='bar', ax=ax, color=['#27ae60' if v > 0 else '#e74c3c' for v in correlations])
ax.set_title('Feature correlation with relevance label')
ax.set_ylabel('Pearson correlation')
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"\nTop 5 positively correlated features:")
print(correlations.head())
print(f"\nTop 5 negatively correlated features:")
print(correlations.tail())

## 3. Evaluation Metrics for Ranking

Before building models, let's implement the metrics we'll use to evaluate them.

### NDCG@K (Normalized Discounted Cumulative Gain)
Measures the quality of the entire top-K ranking. A relevant document at position 1 contributes more than one at position 5.

$$DCG@K = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)}$$

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

where IDCG@K is the DCG of the ideal (perfect) ranking.

### MRR (Mean Reciprocal Rank)
How quickly does the user find the first relevant result?

$$MRR = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{rank_q}$$

where $rank_q$ is the position of the **first** relevant document for query $q$.

In [ ]:
def dcg_at_k(relevances, k):
    """Compute DCG@K for a single ranked list."""
    relevances = np.array(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(relevances) + 2))  # log2(2), log2(3), ...
    return np.sum((2**relevances - 1) / discounts)


def ndcg_at_k(relevances, k):
    """Compute NDCG@K for a single ranked list."""
    dcg = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    if ideal == 0:
        return 0.0  # no relevant docs => undefined, return 0
    return dcg / ideal


def mrr(relevances):
    """Compute Reciprocal Rank for a single ranked list (binary: rel > 0)."""
    for i, rel in enumerate(relevances):
        if rel > 0:
            return 1.0 / (i + 1)
    return 0.0


def evaluate_ranking(df, score_col, k=5):
    """Evaluate a ranking model on a DataFrame with 'qid', 'label', and score_col."""
    ndcgs = []
    mrrs = []
    for qid, group in df.groupby('qid'):
        # Sort by predicted score (descending)
        ranked = group.sort_values(score_col, ascending=False)
        rels = ranked['label'].values
        ndcgs.append(ndcg_at_k(rels, k))
        mrrs.append(mrr(rels))
    return {
        f'NDCG@{k}': np.mean(ndcgs),
        'MRR': np.mean(mrrs),
        'n_queries': len(ndcgs)
    }


# Sanity check: perfect ranking vs random ranking
print('Perfect ranking [2, 1, 0, 0, 0]:  NDCG@5 =', f'{ndcg_at_k([2, 1, 0, 0, 0], 5):.4f}')
print('Reversed ranking [0, 0, 0, 1, 2]: NDCG@5 =', f'{ndcg_at_k([0, 0, 0, 1, 2], 5):.4f}')
print('Random ranking [0, 2, 0, 1, 0]:   NDCG@5 =', f'{ndcg_at_k([0, 2, 0, 1, 0], 5):.4f}')

### Baseline: Random ranking

Let's establish a baseline — what if we rank documents randomly?

In [ ]:
np.random.seed(42)
df_test_eval = df_test.copy()
df_test_eval['random_score'] = np.random.rand(len(df_test_eval))

random_results = evaluate_ranking(df_test_eval, 'random_score', k=5)
print(f"Random baseline: NDCG@5 = {random_results['NDCG@5']:.4f}, MRR = {random_results['MRR']:.4f}")

# Store results for comparison
results = {'Random': random_results}

## 4. Pointwise Approach

**Idea:** Treat ranking as a standard classification/regression problem. Predict the relevance label for each document **independently**, then sort by predicted score.

**Pros:** Simple, can use any classifier/regressor  
**Cons:** Ignores the relative ordering between items — it doesn't know that getting position 1 right matters more than position 10

<img src="https://mermaid.ink/img/eyJjb2RlIjoiZ3JhcGggTFJcbiAgICBBW3F1ZXJ5LWRvYyBmZWF0dXJlc10gLS0-IEJbY2xhc3NpZmllci9yZWdyZXNzb3JdXG4gICAgQiAtLT4gQ1twcmVkaWN0ZWQgcmVsZXZhbmNlIHNjb3JlXVxuICAgIEMgLS0-IERbc29ydCBieSBzY29yZV1cbiIsIm1lcm1haWQiOnsidGhlbWUiOiJkZWZhdWx0In19" width="600" alt="Pointwise approach diagram">

In [ ]:
# Prepare features
feature_cols = [c for c in df_train.columns if c.startswith('f')]

X_train = df_train[feature_cols].values
y_train = df_train['label'].values
X_val = df_val[feature_cols].values
y_val = df_val['label'].values
X_test = df_test[feature_cols].values
y_test = df_test['label'].values

# Normalize features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f'Feature matrix: {X_train_s.shape[0]} samples x {X_train_s.shape[1]} features')

In [ ]:
# Pointwise: Logistic Regression
# We use predict_proba to get continuous scores (not just 0/1/2)
# For multi-class, the score = weighted sum of class probabilities
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train_s, y_train)

# Score = expected relevance = sum(class_prob * class_label)
proba_test = lr.predict_proba(X_test_s)
classes = lr.classes_
df_test_eval['pointwise_lr_score'] = proba_test @ classes

results['Pointwise (LR)'] = evaluate_ranking(df_test_eval, 'pointwise_lr_score', k=5)
print(f"Pointwise (Logistic Regression): NDCG@5 = {results['Pointwise (LR)']['NDCG@5']:.4f}, "
      f"MRR = {results['Pointwise (LR)']['MRR']:.4f}")

In [ ]:
# Pointwise: Gradient Boosting Classifier (stronger model, still pointwise)
gbc = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gbc.fit(X_train, y_train)  # trees don't need scaling

proba_test_gbc = gbc.predict_proba(X_test)
df_test_eval['pointwise_gbc_score'] = proba_test_gbc @ gbc.classes_

results['Pointwise (GBC)'] = evaluate_ranking(df_test_eval, 'pointwise_gbc_score', k=5)
print(f"Pointwise (Gradient Boosting): NDCG@5 = {results['Pointwise (GBC)']['NDCG@5']:.4f}, "
      f"MRR = {results['Pointwise (GBC)']['MRR']:.4f}")

### Discussion: Why is pointwise limited?

The pointwise model treats each document independently. It doesn't know:
- That documents belong to the **same query** and will be compared against each other
- That errors at the **top of the list** are more costly than errors at the bottom
- The relative ordering between items matters more than absolute scores

Can we do better by explicitly learning **relative preferences**?

## 5. Pairwise Approach

**Idea:** For each query, create pairs of documents $(d_i, d_j)$ where $d_i$ is more relevant than $d_j$. Train a model to predict which document in a pair should rank higher.

**Classic methods:** RankNet, RankSVM, BPR (Bayesian Personalized Ranking)

We'll implement a simple pairwise approach:
1. Generate training pairs from within each query
2. Train a classifier on feature differences: $\Delta f = f(d_i) - f(d_j)$
3. At test time, score each document and rank by score

In [ ]:
def generate_pairs(df, feature_cols, max_pairs_per_query=50):
    """Generate pairwise training data: feature differences for pairs with different relevance."""
    X_pairs = []
    y_pairs = []
    
    for qid, group in df.groupby('qid'):
        feats = group[feature_cols].values
        labels = group['label'].values
        
        pairs_generated = 0
        for i in range(len(labels)):
            for j in range(len(labels)):
                if labels[i] > labels[j]:  # doc i is more relevant than doc j
                    X_pairs.append(feats[i] - feats[j])
                    y_pairs.append(1)  # i should rank above j
                    pairs_generated += 1
                    if pairs_generated >= max_pairs_per_query:
                        break
            if pairs_generated >= max_pairs_per_query:
                break
    
    return np.array(X_pairs), np.array(y_pairs)


print('Generating pairwise training data...')
X_pairs, y_pairs = generate_pairs(df_train, feature_cols, max_pairs_per_query=100)
print(f'Generated {len(X_pairs):,} training pairs')
print(f'Positive class ratio: {y_pairs.mean():.2f} (should be 1.0 by construction)')

In [ ]:
# Also add "negative" pairs (swapped) to balance the dataset
X_pairs_full = np.vstack([X_pairs, -X_pairs])  # swap = negate features
y_pairs_full = np.concatenate([np.ones(len(X_pairs)), np.zeros(len(X_pairs))])

# Shuffle
shuffle_idx = np.random.permutation(len(X_pairs_full))
X_pairs_full = X_pairs_full[shuffle_idx]
y_pairs_full = y_pairs_full[shuffle_idx]

print(f'Balanced pairwise dataset: {len(X_pairs_full):,} pairs')

# Train a logistic regression on pairs
pair_model = LogisticRegression(max_iter=1000, C=1.0)
pair_model.fit(X_pairs_full, y_pairs_full)
print(f'Pairwise model trained. Train accuracy: {pair_model.score(X_pairs_full, y_pairs_full):.4f}')

In [ ]:
# At test time: score each document using the pairwise model's weights
# The learned weight vector w tells us: if w . (fi - fj) > 0, doc i ranks above doc j
# So for individual scoring: score(d) = w . f(d)
# This is because: w.(fi - fj) = w.fi - w.fj = score(i) - score(j)

df_test_eval['pairwise_score'] = X_test @ pair_model.coef_.flatten() + pair_model.intercept_

results['Pairwise (LR)'] = evaluate_ranking(df_test_eval, 'pairwise_score', k=5)
print(f"Pairwise (Logistic Regression): NDCG@5 = {results['Pairwise (LR)']['NDCG@5']:.4f}, "
      f"MRR = {results['Pairwise (LR)']['MRR']:.4f}")

### Discussion: Pairwise pros and cons

**Improvement over pointwise?** The pairwise approach explicitly learns *relative preferences* between items in the same query. This often helps.

**Limitation:** It treats all pairs equally — swapping positions 1 and 2 is penalized the same as swapping positions 99 and 100. But for the user, getting position 1 right matters **much more**!

This is where **listwise** methods shine — they directly optimize metrics like NDCG that weight top positions more heavily.

## 6. Listwise Approach: LambdaMART

**LambdaMART** is the industry workhorse of LTR. It combines:
- **MART** (Multiple Additive Regression Trees) = gradient boosted decision trees
- **LambdaRank** gradients that approximate NDCG optimization

The key insight: when computing gradients for a pair of documents, LambdaMART **scales** the gradient by the change in NDCG that would result from swapping those two documents. This means:
- Swapping positions 1 and 2 → large gradient (big NDCG impact)
- Swapping positions 99 and 100 → tiny gradient (negligible NDCG impact)

**LightGBM** provides an excellent implementation via `objective='lambdarank'`.

### Why LambdaMART is still king
> *"LambdaMART remains a highly effective, efficient, and widely deployed algorithm, particularly with well-engineered features. It serves as a crucial component in countless search and recommendation systems and remains a formidable baseline."*

In [ ]:
# Prepare data in LightGBM format
# LightGBM ranker needs a "group" array: number of documents per query

def get_groups(df):
    """Get group sizes for LightGBM ranker."""
    return df.groupby('qid').size().values

train_groups = get_groups(df_train)
val_groups = get_groups(df_val)
test_groups = get_groups(df_test)

print(f'Train groups: {len(train_groups)} queries, group sizes: {train_groups[:10]}...')

# Create LightGBM datasets
lgb_train = lgb.Dataset(X_train, label=y_train, group=train_groups)
lgb_val = lgb.Dataset(X_val, label=y_val, group=val_groups, reference=lgb_train)

In [ ]:
# Train LambdaMART
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'eval_at': [5, 10],           # Evaluate NDCG@5 and NDCG@10
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 10,
    'max_depth': -1,              # No limit
    'feature_fraction': 0.8,      # Use 80% of features per tree
    'bagging_fraction': 0.8,      # Use 80% of data per tree
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

print('Training LambdaMART...')
callbacks = [lgb.log_evaluation(period=50)]
model_lambdamart = lgb.train(
    params,
    lgb_train,
    num_boost_round=300,
    valid_sets=[lgb_val],
    valid_names=['validation'],
    callbacks=callbacks
)
print('Done!')

In [ ]:
# Evaluate LambdaMART
df_test_eval['lambdamart_score'] = model_lambdamart.predict(X_test)

results['LambdaMART'] = evaluate_ranking(df_test_eval, 'lambdamart_score', k=5)
print(f"LambdaMART (LightGBM): NDCG@5 = {results['LambdaMART']['NDCG@5']:.4f}, "
      f"MRR = {results['LambdaMART']['MRR']:.4f}")

In [ ]:
# Feature importance from LambdaMART
importance = model_lambdamart.feature_importance(importance_type='gain')
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
feat_imp.head(15).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 Features by Importance (LambdaMART)')
ax.set_ylabel('Gain')
plt.tight_layout()
plt.show()

## 7. Comparison of All Approaches

In [ ]:
# Summary table
summary = pd.DataFrame(results).T[['NDCG@5', 'MRR']]
summary = summary.sort_values('NDCG@5', ascending=False)
print(summary.to_string(float_format='%.4f'))
print()

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#95a5a6', '#e74c3c', '#e67e22', '#3498db', '#27ae60']
order = ['Random', 'Pointwise (LR)', 'Pointwise (GBC)', 'Pairwise (LR)', 'LambdaMART']
order = [o for o in order if o in summary.index]

for ax, metric in zip(axes, ['NDCG@5', 'MRR']):
    vals = [summary.loc[m, metric] for m in order]
    bars = ax.bar(range(len(order)), vals, color=colors[:len(order)])
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=30, ha='right')
    ax.set_ylabel(metric)
    ax.set_title(metric)
    # Add value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Learning to Rank: Method Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Deeper Dive: Tuning LambdaMART

Let's experiment with hyperparameters to see their effect.

In [ ]:
# Experiment: Effect of number of trees
n_trees_list = [10, 25, 50, 100, 200, 300, 500]
ndcg_by_trees = []

for n_trees in n_trees_list:
    m = lgb.train(params, lgb_train, num_boost_round=n_trees,
                  valid_sets=[lgb_val], valid_names=['val'],
                  callbacks=[lgb.log_evaluation(period=0)])
    df_test_eval['_tmp_score'] = m.predict(X_test)
    res = evaluate_ranking(df_test_eval, '_tmp_score', k=5)
    ndcg_by_trees.append(res['NDCG@5'])
    print(f'  n_trees={n_trees:3d} -> NDCG@5 = {res["NDCG@5"]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_trees_list, ndcg_by_trees, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Number of trees')
ax.set_ylabel('NDCG@5 (test)')
ax.set_title('LambdaMART: Effect of ensemble size')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Experiment: Effect of num_leaves (tree complexity)
leaves_list = [5, 10, 15, 31, 50, 80]
ndcg_by_leaves = []

for n_leaves in leaves_list:
    p = params.copy()
    p['num_leaves'] = n_leaves
    m = lgb.train(p, lgb_train, num_boost_round=200,
                  valid_sets=[lgb_val], valid_names=['val'],
                  callbacks=[lgb.log_evaluation(period=0)])
    df_test_eval['_tmp_score'] = m.predict(X_test)
    res = evaluate_ranking(df_test_eval, '_tmp_score', k=5)
    ndcg_by_leaves.append(res['NDCG@5'])
    print(f'  num_leaves={n_leaves:3d} -> NDCG@5 = {res["NDCG@5"]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(leaves_list, ndcg_by_leaves, 's-', color='#e74c3c', linewidth=2, markersize=8)
ax.set_xlabel('num_leaves')
ax.set_ylabel('NDCG@5 (test)')
ax.set_title('LambdaMART: Effect of tree complexity')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Understanding Position Bias

In real-world systems, users click on items shown at the **top** more often — regardless of true relevance. This is **position bias**.

If we naively train on click data, the model learns to reinforce whatever ordering was already there, creating a **feedback loop**.

Let's simulate this effect and see how it impacts model training.

In [ ]:
# Simulate position bias: items at higher positions get artificially inflated clicks
def simulate_position_bias(df, position_decay=0.7):
    """Simulate biased click data where higher positions get more clicks.
    
    Model: P(click) = P(relevant) * P(examined)
    where P(examined) decays with position: position_decay^(rank-1)
    """
    df_biased = df.copy()
    biased_labels = []
    
    for qid, group in df_biased.groupby('qid'):
        n = len(group)
        # Simulate random initial ordering
        positions = np.random.permutation(n)
        # Examination probability decays with position
        exam_prob = position_decay ** positions
        # Click = relevant AND examined
        true_rel = group['label'].values
        click_prob = (true_rel > 0).astype(float) * exam_prob
        # Sample clicks
        clicks = (np.random.rand(n) < click_prob).astype(int)
        biased_labels.extend(clicks)
    
    df_biased['biased_label'] = biased_labels
    return df_biased


np.random.seed(42)
df_train_biased = simulate_position_bias(df_train)

print('True labels vs biased clicks:')
print(f'  True relevant docs:  {(df_train["label"] > 0).sum():,}')
print(f'  Biased clicks:       {df_train_biased["biased_label"].sum():,}')
print(f'  Click rate:          {df_train_biased["biased_label"].mean():.3f}')

In [ ]:
# Train on biased data vs true labels
# Model on TRUE labels (our standard model)
true_ndcg = results['LambdaMART']['NDCG@5']

# Model on BIASED clicks
y_biased = df_train_biased['biased_label'].values
lgb_biased = lgb.Dataset(X_train, label=y_biased, group=train_groups)

model_biased = lgb.train(params, lgb_biased, num_boost_round=200,
                         callbacks=[lgb.log_evaluation(period=0)])

df_test_eval['biased_model_score'] = model_biased.predict(X_test)
biased_result = evaluate_ranking(df_test_eval, 'biased_model_score', k=5)

print(f'Model trained on TRUE labels:   NDCG@5 = {true_ndcg:.4f}')
print(f'Model trained on BIASED clicks: NDCG@5 = {biased_result["NDCG@5"]:.4f}')
print(f'\nDegradation from bias: {true_ndcg - biased_result["NDCG@5"]:.4f}')
print('\nThis shows why debiasing is important in real-world LTR systems!')

### Debiasing strategies (overview)

In practice, you can mitigate position bias by:

1. **Position as a feature** — let the model learn to discount position effects
2. **Inverse Propensity Scoring (IPS)** — re-weight training samples by $1/P(\text{examined})$
3. **Randomization** — occasionally shuffle results to collect unbiased data
4. **Position-aware models** — DLA (Dual Learning Algorithm) jointly learns relevance and propensity

**Important caveat:** Recent research on Baidu search logs showed that standard ULTR methods improved click prediction but did not consistently improve actual ranking quality (NDCG). Simple strategies (position feature) often work well enough.

## 10. From Search Ranking to Personalized Recommendations

Everything we've done applies directly to **recommendation systems** and **personalized query suggestion** (like the AOL4PS assignment). The mapping is:

| Search LTR | Recommendations | Query Suggestion |
|------------|----------------|------------------|
| Query | User (or user context) | User's search history |
| Document | Item to recommend | Candidate query to suggest |
| Relevance label | Click / purchase | User actually searched this |
| Features | Query-doc features (BM25, etc.) | User-query features (history match, popularity, embeddings) |

### The AOL4PS Dataset

The [AOL4PS dataset](https://doi.org/10.11922/sciencedb.j00104.00093) contains:
- **Users** (anonymized IDs) with search query histories
- **Queries** with text
- **Documents** with URLs and titles
- **Clicks** with position information
- Pre-defined temporal train/val/test splits

For a personalized query suggestion task, you would:
1. Treat each user's search session as the "query" context
2. Candidate queries (from a pool) are the "documents" to rank
3. Features could include: text similarity (embeddings), user history overlap, query popularity, time features
4. Use LambdaMART or a two-tower model to rank candidate suggestions

## 11. Practical Exercise: Feature Engineering

Feature engineering is often **more impactful** than model choice. Let's experiment with the MQ2008 features.

In [ ]:
# Exercise: What happens if we use only the top-K most important features?
top_features = feat_imp.head(10).index.tolist()
print(f'Top 10 features: {top_features}')

X_train_top = df_train[top_features].values
X_val_top = df_val[top_features].values
X_test_top = df_test[top_features].values

lgb_train_top = lgb.Dataset(X_train_top, label=y_train, group=train_groups)
lgb_val_top = lgb.Dataset(X_val_top, label=y_val, group=val_groups, reference=lgb_train_top)

model_top = lgb.train(params, lgb_train_top, num_boost_round=200,
                      valid_sets=[lgb_val_top], valid_names=['val'],
                      callbacks=[lgb.log_evaluation(period=0)])

df_test_eval['top10_score'] = model_top.predict(X_test_top)
top10_result = evaluate_ranking(df_test_eval, 'top10_score', k=5)

print(f'\nAll {len(feature_cols)} features: NDCG@5 = {results["LambdaMART"]["NDCG@5"]:.4f}')
print(f'Top 10 features only:    NDCG@5 = {top10_result["NDCG@5"]:.4f}')
print(f'\nKey insight: a handful of well-chosen features often captures most of the performance!')

In [ ]:
# Exercise: Create interaction features (cross-features)
# Let's create ratios and products of top features
top2 = feat_imp.head(2).index.tolist()

for split_name, df_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    # Product interaction
    df_split[f'{top2[0]}_x_{top2[1]}'] = df_split[top2[0]] * df_split[top2[1]]
    # Ratio (with small epsilon to avoid division by zero)
    df_split[f'{top2[0]}_div_{top2[1]}'] = df_split[top2[0]] / (df_split[top2[1]].abs() + 1e-6)

new_feature_cols = feature_cols + [f'{top2[0]}_x_{top2[1]}', f'{top2[0]}_div_{top2[1]}']

X_train_new = df_train[new_feature_cols].values
X_val_new = df_val[new_feature_cols].values
X_test_new = df_test[new_feature_cols].values

lgb_train_new = lgb.Dataset(X_train_new, label=y_train, group=train_groups)
lgb_val_new = lgb.Dataset(X_val_new, label=y_val, group=val_groups, reference=lgb_train_new)

model_new = lgb.train(params, lgb_train_new, num_boost_round=200,
                      valid_sets=[lgb_val_new], valid_names=['val'],
                      callbacks=[lgb.log_evaluation(period=0)])

df_test_eval['new_feat_score'] = model_new.predict(X_test_new)
new_result = evaluate_ranking(df_test_eval, 'new_feat_score', k=5)

print(f'Original features:          NDCG@5 = {results["LambdaMART"]["NDCG@5"]:.4f}')
print(f'With interaction features:   NDCG@5 = {new_result["NDCG@5"]:.4f}')
print(f'\nNote: Trees can learn interactions automatically, so manual crosses may not always help.')
print('But for linear models or two-tower architectures, explicit cross-features are crucial!')

## 12. Per-Query Analysis: Where Does the Model Fail?

Understanding failure modes is essential for improving ranking systems.

In [ ]:
# Compute per-query NDCG@5 for LambdaMART
per_query_ndcg = []
per_query_sizes = []
per_query_n_rel = []

for qid, group in df_test_eval.groupby('qid'):
    ranked = group.sort_values('lambdamart_score', ascending=False)
    rels = ranked['label'].values
    n = ndcg_at_k(rels, 5)
    per_query_ndcg.append(n)
    per_query_sizes.append(len(group))
    per_query_n_rel.append((group['label'] > 0).sum())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of NDCG@5
axes[0].hist(per_query_ndcg, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('NDCG@5')
axes[0].set_ylabel('Number of queries')
axes[0].set_title('Distribution of per-query NDCG@5')
axes[0].axvline(x=np.mean(per_query_ndcg), color='red', linestyle='--', label=f'Mean={np.mean(per_query_ndcg):.3f}')
axes[0].legend()

# NDCG vs list size
axes[1].scatter(per_query_sizes, per_query_ndcg, alpha=0.3, s=20, color='steelblue')
axes[1].set_xlabel('List size (# docs)')
axes[1].set_ylabel('NDCG@5')
axes[1].set_title('NDCG@5 vs List Size')

# NDCG vs number of relevant docs
axes[2].scatter(per_query_n_rel, per_query_ndcg, alpha=0.3, s=20, color='#e74c3c')
axes[2].set_xlabel('Number of relevant docs')
axes[2].set_ylabel('NDCG@5')
axes[2].set_title('NDCG@5 vs # Relevant Docs')

plt.tight_layout()
plt.show()

print(f'Queries with perfect ranking (NDCG@5 = 1.0): '
      f'{sum(1 for n in per_query_ndcg if n >= 0.999)}/{len(per_query_ndcg)}')
print(f'Queries with zero NDCG@5: '
      f'{sum(1 for n in per_query_ndcg if n < 0.001)}/{len(per_query_ndcg)}')

## 13. Bonus: XGBoost Comparison

XGBoost also supports LambdaMART. Let's compare.

In [ ]:
try:
    import xgboost as xgb
    
    # XGBoost requires cumulative group boundaries
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(train_groups)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(val_groups)
    dtest = xgb.DMatrix(X_test, label=y_test)
    dtest.set_group(test_groups)
    
    xgb_params = {
        'objective': 'rank:ndcg',
        'eval_metric': 'ndcg@5',
        'eta': 0.05,
        'max_depth': 6,
        'min_child_weight': 10,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42,
        'verbosity': 0
    }
    
    model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=200,
                          evals=[(dval, 'val')], verbose_eval=50)
    
    df_test_eval['xgb_score'] = model_xgb.predict(dtest)
    results['XGBoost LambdaMART'] = evaluate_ranking(df_test_eval, 'xgb_score', k=5)
    print(f"\nXGBoost: NDCG@5 = {results['XGBoost LambdaMART']['NDCG@5']:.4f}, "
          f"MRR = {results['XGBoost LambdaMART']['MRR']:.4f}")
    
except ImportError:
    print('XGBoost not installed. Install with: pip install xgboost')
    print('Skipping this section.')

## 14. Final Summary and Takeaways

In [ ]:
# Final results table
summary_final = pd.DataFrame(results).T[['NDCG@5', 'MRR']]
summary_final = summary_final.sort_values('NDCG@5', ascending=False)
print('='*50)
print('     FINAL RESULTS COMPARISON')
print('='*50)
print(summary_final.to_string(float_format='%.4f'))
print('='*50)

## Key Takeaways

1. **Pointwise < Pairwise < Listwise** — each approach captures more ranking information
2. **LambdaMART is the workhorse** — gradient boosted trees with LambdaRank gradients remain the strongest practical baseline
3. **Feature engineering matters more than model choice** — a good set of features with LambdaMART often beats a fancy model with poor features
4. **Position bias is real** — training on biased clicks degrades ranking quality; use debiasing strategies
5. **Evaluation metrics:** NDCG@K rewards the whole top-K quality; MRR focuses on the first relevant result

## Connection to the Assignment (Personalized Query Suggestion)

For the [AOL4PS dataset](https://doi.org/10.11922/sciencedb.j00104.00093) assignment:
- **Next query prediction** = ranking candidate queries for a user
- Features: text embeddings (for semantic similarity), user history overlap, query frequency, temporal features
- Evaluation: use **fuzzy semantic metrics** (cosine similarity of embeddings) instead of exact string match
- The LTR pipeline we built here transfers directly — just swap the features!

## Further Reading

- [LambdaMART Explained](https://www.shaped.ai/blog/lambdamart-explained-the-workhorse-of-learning-to-rank) — Shaped Blog
- [LightGBM LTR in Python](https://forecastegy.com/posts/lightgbm-learning-to-rank-python/) — Forecastegy
- [Evaluating Recommendation Systems (mAP, MRR, NDCG)](https://www.shaped.ai/blog/evaluating-recommendation-systems-map-mmr-ndcg) — Shaped Blog
- [allRank: PyTorch LTR Framework](https://github.com/allegro/allRank)
- [Unbiased LTR Meets Reality (SIGIR 2024)](https://staff.fnwi.uva.nl/m.derijke/wp-content/papercite-data/pdf/hager-2024-unbiased.pdf)